# Download and process WHACS offshore waves

Download WHACS bulk and partition spectra, remap them onto metamodel goals, and
write the processed zarr used by dynamic forcing and hindcast reconstruction.

**Prerequisites:** ``04_Select_Goals_by_Hindcasts.ipynb`` (``Samoa_goals.json``).

## Configuration


In [1]:
import xarray as xr

whacs_data = xr.open_dataset("inputs/WHACS/processed/Samoa_WHACS_ALL.zarr")
whacs_data = whacs_data.assign({"tp": 1.0 / whacs_data["fp"]})
whacs_data["tp"].attrs = {
    "long_name": "Wave peak period",
    "units": "s",
}
whacs_data

<xarray.Dataset> Size: 1GB
Dimensions:                      (time: 394464, seapoint: 11)
Coordinates:
  * time                         (time) datetime64[ns] 3MB 1979-01-01 ... 202...
  * seapoint                     (seapoint) int32 44B 969426 969776 ... 971049
Data variables: (12/42)
    cge                          (time, seapoint) float64 35MB ...
    dir                          (time, seapoint) float64 35MB ...
    dp                           (time, seapoint) float64 35MB ...
    fp                           (time, seapoint) float64 35MB 0.06 ... 0.074
    hs                           (time, seapoint) float64 35MB ...
    latitude                     (time, seapoint) float32 17MB ...
    ...                           ...
    spr                          (time, seapoint) float64 35MB ...
    t01                          (time, seapoint) float64 35MB ...
    t02                          (time, seapoint) float64 35MB ...
    uwnd                         (time, seapoint) float64 35MB ...
    vwnd                         (time, seapoint) float64 35MB ...
    tp                           (time, seapoint) float64 35MB 16.67 ... 13.51
Attributes: (12/59)
    Conventions:                     CF-1.11 ACDD-1.3
    title:                           Wave Hindcast for the Australian Climate...
    summary:                         Wave Hindcast for the Australian Climate...
    keywords:                        WHACS, ACS, waves, ocean, coast, WWIII, ...
    id:                              BoM-CSIRO-WHACS-001
    naming_authority:                au.csiro
    ...                              ...
    geospatial_lon_max:              179.9375
    geospatial_vertical_positive:    up
    time_coverage_start:             1979-01-01T00:00:00
    time_coverage_end:               1979-01-31T23:00:00
    time_coverage_duration:          P30DT23H0M0S
    time_coverage_resolution:        P0DT1H0M0S

In [2]:
ordered_seapoints_for_goals = [
    971044,
    971049,
    970816,
    970701,
    970364,
    970142,
    969426,
    969781,
    969776,
    970007,
    970465,
]

whacs_data = whacs_data.sel(seapoint=ordered_seapoints_for_goals)[
    [
        "hs",
        "tp",
        "dir",
        "spr",
        "phs0",
        "ptp0",
        "pdir0",
        "pspr0",
        "phs1",
        "ptp1",
        "pdir1",
        "pspr1",
        "phs2",
        "ptp2",
        "pdir2",
        "pspr2",
        "phs3",
        "ptp3",
        "pdir3",
        "pspr3",
    ]
].rename({"seapoint": "goal"})
whacs_data["goal"] = range(1, 12)
whacs_data = whacs_data.assign_coords(
    {"seapoint": ("goal", ordered_seapoints_for_goals)}
)
whacs_data.to_zarr("inputs/WHACS/processed/Samoa_WHACS.zarr")

/home/tausiaj/miniforge3/envs/bluemath-311-dev/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [7]:
xr.open_zarr("inputs/WHACS/processed/Samoa_WHACS.zarr")

<xarray.Dataset> Size: 697MB
Dimensions:   (time: 394464, goal: 11)
Coordinates:
  * time      (time) datetime64[ns] 3MB 1979-01-01 ... 2023-12-31T23:00:00
  * goal      (goal) int64 88B 1 2 3 4 5 6 7 8 9 10 11
    seapoint  (goal) int64 88B dask.array<chunksize=(11,), meta=np.ndarray>
Data variables: (12/20)
    dir       (time, goal) float64 35MB dask.array<chunksize=(394464, 11), meta=np.ndarray>
    hs        (time, goal) float64 35MB dask.array<chunksize=(394464, 11), meta=np.ndarray>
    pdir0     (time, goal) float64 35MB dask.array<chunksize=(394464, 11), meta=np.ndarray>
    pdir1     (time, goal) float64 35MB dask.array<chunksize=(394464, 11), meta=np.ndarray>
    pdir2     (time, goal) float64 35MB dask.array<chunksize=(394464, 11), meta=np.ndarray>
    pdir3     (time, goal) float64 35MB dask.array<chunksize=(394464, 11), meta=np.ndarray>
    ...        ...
    ptp0      (time, goal) float64 35MB dask.array<chunksize=(394464, 11), meta=np.ndarray>
    ptp1      (time, goal) float64 35MB dask.array<chunksize=(394464, 11), meta=np.ndarray>
    ptp2      (time, goal) float64 35MB dask.array<chunksize=(394464, 11), meta=np.ndarray>
    ptp3      (time, goal) float64 35MB dask.array<chunksize=(394464, 11), meta=np.ndarray>
    spr       (time, goal) float64 35MB dask.array<chunksize=(394464, 11), meta=np.ndarray>
    tp        (time, goal) float64 35MB dask.array<chunksize=(49308, 2), meta=np.ndarray>
Attributes: (12/59)
    Conventions:                     CF-1.11 ACDD-1.3
    title:                           Wave Hindcast for the Australian Climate...
    summary:                         Wave Hindcast for the Australian Climate...
    keywords:                        WHACS, ACS, waves, ocean, coast, WWIII, ...
    id:                              BoM-CSIRO-WHACS-001
    naming_authority:                au.csiro
    ...                              ...
    geospatial_lon_max:              179.9375
    geospatial_vertical_positive:    up
    time_coverage_start:             1979-01-01T00:00:00
    time_coverage_end:               1979-01-31T23:00:00
    time_coverage_duration:          P30DT23H0M0S
    time_coverage_resolution:        P0DT1H0M0S